# DACON PANNs + HTDemucs + XLS-R TCM 제출 패키지 생성기

DACON 공식 베이스라인의 아래 흐름을 유지하면서 `DF-Arena 1B`만 직접 학습한 `exp06_xlsr_tcm/best.pt`로 교체합니다.

```text
INPUT AUDIO
├── PANNs Cnn14 ───────────────> VOICE_PRESENT_PROB / MUSIC_PRESENT_PROB
└── HTDemucs
    ├── vocals ────────────────> XLS-R TCM ──> VOICE_FAKE_PROB
    └── accompaniment ─────────> XLS-R TCM ──> MUSIC_FAKE_PROB

FILE_FAKE_PROB = max(VP × VF, MP × MF)
```

이 노트북은 학습 노트북이 아니라 **이미 학습된 checkpoint와 DACON 공식 베이스라인 자산으로 `submit.zip`을 만드는 패키징 노트북**입니다.

준비물:

1. DACON 코드 공유에서 다운로드한 원본 베이스라인 `submit.zip`
2. 기존 전체 학습 노트북이 저장한 `deepvoice_for/runs/exp06_xlsr_tcm/best.pt`

중요: FoR로 학습한 XLS-R TCM은 음성 real/fake 이진 분류기입니다. 동일 모델을 accompaniment에도 적용하는 것은 요청한 구조를 구현하지만, 음악 생성 탐지에 특화해 학습된 것은 아니므로 리더보드 성능은 보장되지 않습니다.


## 1. Google Drive 연결과 경로 설정

`BASELINE_ZIP_PATH`만 실제 원본 베이스라인 ZIP 위치에 맞게 바꾸세요. 출력은 별도 폴더의 `submit.zip`에 저장해 기존 파일을 덮어쓰지 않습니다.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive")

# DACON 코드 공유 14153에서 다운로드한 원본 베이스라인 ZIP
BASELINE_ZIP_PATH = DRIVE_ROOT / "dacon_baseline" / "submit.zip"

# 기존 전체 학습 노트북이 저장한 XLS-R TCM best checkpoint
XLSR_TCM_CHECKPOINT = (
    DRIVE_ROOT / "deepvoice_for" / "runs" / "exp06_xlsr_tcm" / "best.pt"
)

OUTPUT_DIR = DRIVE_ROOT / "deepvoice_for" / "dacon_htdemucs_xlsr_tcm"
OUTPUT_ZIP_PATH = OUTPUT_DIR / "submit.zip"
BUILD_ROOT = Path("/content/dacon_xlsr_tcm_build")

print("baseline:", BASELINE_ZIP_PATH)
print("checkpoint:", XLSR_TCM_CHECKPOINT)
print("output:", OUTPUT_ZIP_PATH)


## 2. 입력 파일과 checkpoint 검사

checkpoint의 모델 종류, 라벨 규약, 주요 state key를 먼저 확인합니다. 학습 코드의 규약은 `0=fake`, `1=real`이며 제출 코드가 이 정보를 읽어 FAKE class index를 자동 결정합니다.


In [ ]:
import ast
import json
import shutil
import tempfile
import zipfile

import torch

if not BASELINE_ZIP_PATH.is_file():
    raise FileNotFoundError(
        f"원본 DACON 베이스라인 ZIP을 찾지 못했습니다: {BASELINE_ZIP_PATH}"
    )
if not XLSR_TCM_CHECKPOINT.is_file():
    raise FileNotFoundError(
        f"XLS-R TCM best checkpoint를 찾지 못했습니다: {XLSR_TCM_CHECKPOINT}"
    )

checkpoint = torch.load(
    XLSR_TCM_CHECKPOINT, map_location="cpu", weights_only=False
)
checkpoint_config = checkpoint.get("config", {})
label_convention = checkpoint.get(
    "label_convention", {"0": "fake", "1": "real"}
)
if checkpoint_config.get("model") != "xlsr_tcm":
    raise ValueError(
        f"exp06_xlsr_tcm checkpoint가 아닙니다: {checkpoint_config}"
    )
if "model_state" not in checkpoint:
    raise KeyError("checkpoint에 model_state가 없습니다.")

state_keys = list(checkpoint["model_state"])
for required_prefix in ("ssl.", "projection.", "tcm.", "pool.", "head."):
    if not any(key.startswith(required_prefix) for key in state_keys):
        raise ValueError(f"checkpoint에 {required_prefix} 계열 weight가 없습니다.")

fake_indices = [
    int(index)
    for index, name in label_convention.items()
    if str(name).strip().lower() == "fake"
]
if len(fake_indices) != 1:
    raise ValueError(f"fake class index를 하나로 결정할 수 없습니다: {label_convention}")

print("checkpoint epoch:", checkpoint.get("epoch"))
print("robust_score:", checkpoint.get("robust_score"))
print("label convention:", label_convention)
print("fake class index:", fake_indices[0])
print("state tensors:", len(state_keys))


## 3. 원본 베이스라인에서 PANNs·HTDemucs 자산 찾기

ZIP 전체를 복사하지 않습니다. `model/panns`와 `model/htdemucs`만 가져오므로 DF-Arena 1B weight와 Python module은 최종 ZIP에 들어가지 않습니다.


In [ ]:
def safe_extract(archive_path, destination):
    destination = destination.resolve()
    with zipfile.ZipFile(archive_path) as archive:
        for member in archive.infolist():
            member_path = Path(member.filename)
            if member_path.is_absolute() or ".." in member_path.parts:
                raise ValueError(f"안전하지 않은 ZIP 경로: {member.filename}")
            resolved = (destination / member_path).resolve()
            if destination not in resolved.parents and resolved != destination:
                raise ValueError(f"추출 범위를 벗어난 ZIP 경로: {member.filename}")
        archive.extractall(destination)


def find_unique_asset(root, directory_name, required_files):
    candidates = []
    for candidate in root.rglob(directory_name):
        if candidate.is_dir() and all((candidate / name).is_file() for name in required_files):
            candidates.append(candidate)
    if len(candidates) != 1:
        raise RuntimeError(
            f"{directory_name} 자산 폴더가 정확히 1개여야 합니다. 발견: {candidates}"
        )
    return candidates[0]


extract_root = BUILD_ROOT / "baseline_extracted"
stage_root = BUILD_ROOT / "submit"

resolved_build_root = BUILD_ROOT.resolve()
if str(resolved_build_root) != "/content/dacon_xlsr_tcm_build":
    raise RuntimeError(f"예상하지 않은 작업 경로: {resolved_build_root}")
if BUILD_ROOT.exists():
    shutil.rmtree(BUILD_ROOT)
extract_root.mkdir(parents=True)

safe_extract(BASELINE_ZIP_PATH, extract_root)

panns_source = find_unique_asset(
    extract_root,
    "panns",
    ["Cnn14_mAP=0.431.pth", "class_labels_indices.csv", "component_labels.json"],
)
htdemucs_source = find_unique_asset(extract_root, "htdemucs", [])
htdemucs_files = [path for path in htdemucs_source.rglob("*") if path.is_file()]
if not htdemucs_files:
    raise RuntimeError(f"HTDemucs 폴더가 비어 있습니다: {htdemucs_source}")

print("PANNs source:", panns_source)
print("HTDemucs source:", htdemucs_source)
print("HTDemucs files:", len(htdemucs_files))


## 4. XLS-R 로컬 config와 제출 `script.py` 준비

평가 서버는 추론 중 인터넷에 연결할 수 없으므로 XLS-R 구조 config를 ZIP 안에 저장합니다. 실제 XLS-R weight는 `best.pt`의 `model_state`에 포함되어 있습니다.


In [ ]:
SCRIPT_SOURCE = '#!/usr/bin/env python3\n"""DACON deepvoice inference: PANNs + HTDemucs + a user-trained XLS-R TCM."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport gc\nimport json\nimport os\nimport shutil\nimport sys\nfrom pathlib import Path\n\n# Evaluation is offline after dependency installation.\nos.environ["HF_HUB_OFFLINE"] = "1"\nos.environ["TRANSFORMERS_OFFLINE"] = "1"\nos.environ["HF_DATASETS_OFFLINE"] = "1"\nos.environ["TOKENIZERS_PARALLELISM"] = "false"\nsys.dont_write_bytecode = True\n\nimport librosa\nimport numpy as np\nimport torch\nimport torch.nn as nn\nimport torchaudio\nfrom demucs.apply import apply_model\nfrom demucs.pretrained import get_model\nfrom demucs.separate import load_track\nfrom torch.utils.data import DataLoader, TensorDataset\nfrom tqdm import tqdm\nfrom transformers import AutoConfig, AutoModel\n\n\nBASE_DIR = Path(__file__).resolve().parent\nMODEL_DIR = BASE_DIR / "model"\nXLSR_TCM_DIR = MODEL_DIR / "xlsr_tcm"\nXLSR_CONFIG_DIR = MODEL_DIR / "xlsr_config"\nHTDEMUCS_DIR = MODEL_DIR / "htdemucs"\nPANNS_DIR = MODEL_DIR / "panns"\n\nDEFAULT_TEST_DIR = Path("data") / "test"\nDEFAULT_SAMPLE_SUBMISSION = Path("data") / "sample_submission.csv"\nDEFAULT_OUTPUT_PATH = Path("output") / "submission.csv"\n\nAUDIO_SAMPLE_RATE = 16_000\nPANNS_SAMPLE_RATE = 32_000\nSEGMENT_SAMPLES = 64_600\nSILENCE_RMS = 1e-5\n\nPREDICTION_COLUMNS = [\n    "FILE_FAKE_PROB",\n    "VOICE_FAKE_PROB",\n    "MUSIC_FAKE_PROB",\n    "VOICE_PRESENT_PROB",\n    "MUSIC_PRESENT_PROB",\n]\nSUPPORTED_AUDIO_EXTENSIONS = {\n    ".aac",\n    ".flac",\n    ".m4a",\n    ".mp3",\n    ".ogg",\n    ".opus",\n    ".wav",\n    ".wma",\n}\n\n\ndef parse_arguments(argv=None):\n    parser = argparse.ArgumentParser(\n        description="Run PANNs + HTDemucs + XLS-R TCM inference."\n    )\n    parser.add_argument("--test-dir", type=Path, default=DEFAULT_TEST_DIR)\n    parser.add_argument(\n        "--sample-submission", type=Path, default=DEFAULT_SAMPLE_SUBMISSION\n    )\n    parser.add_argument("--output", type=Path, default=DEFAULT_OUTPUT_PATH)\n    parser.add_argument("--device", choices=["cuda", "cpu"], default="cuda")\n    return parser.parse_args(argv)\n\n\ndef select_device(device_name):\n    if device_name == "cuda" and not torch.cuda.is_available():\n        raise RuntimeError("CUDA is not available")\n    return torch.device(device_name)\n\n\ndef find_audio_files(test_dir):\n    if not test_dir.is_dir():\n        raise FileNotFoundError(f"Test directory not found: {test_dir}")\n\n    audio_files = sorted(\n        (\n            path\n            for path in test_dir.iterdir()\n            if path.is_file() and path.suffix.lower() in SUPPORTED_AUDIO_EXTENSIONS\n        ),\n        key=lambda path: path.stem,\n    )\n    if not audio_files:\n        raise FileNotFoundError(f"No audio files found in {test_dir}")\n\n    audio_ids = [path.stem for path in audio_files]\n    if len(audio_ids) != len(set(audio_ids)):\n        raise ValueError("Audio IDs must be unique")\n    return audio_files\n\n\ndef read_sample_submission(csv_path):\n    if not csv_path.is_file():\n        raise FileNotFoundError(f"Sample submission not found: {csv_path}")\n\n    with csv_path.open("r", encoding="utf-8-sig", newline="") as file:\n        reader = csv.DictReader(file)\n        column_names = reader.fieldnames\n        rows = list(reader)\n\n    if column_names is None or not rows:\n        raise ValueError(f"Invalid sample submission: {csv_path}")\n\n    required_columns = ["ID"] + PREDICTION_COLUMNS\n    missing_columns = [name for name in required_columns if name not in column_names]\n    if missing_columns:\n        raise ValueError(f"Sample submission is missing columns: {missing_columns}")\n\n    seen_ids = set()\n    for row in rows:\n        audio_id = str(row["ID"]).strip()\n        if not audio_id:\n            raise ValueError("Sample submission contains an empty ID")\n        if audio_id in seen_ids:\n            raise ValueError(f"Duplicate ID in sample submission: {audio_id}")\n        seen_ids.add(audio_id)\n        row["ID"] = audio_id\n    return column_names, rows\n\n\ndef order_audio_files(audio_files, submission_rows):\n    audio_by_id = {path.stem: path for path in audio_files}\n    submission_ids = [row["ID"] for row in submission_rows]\n    missing_ids = [audio_id for audio_id in submission_ids if audio_id not in audio_by_id]\n    extra_ids = [audio_id for audio_id in audio_by_id if audio_id not in submission_ids]\n    if missing_ids or extra_ids:\n        raise ValueError(\n            "Test audio and sample submission IDs do not match. "\n            f"Missing: {missing_ids[:5]}, Extra: {extra_ids[:5]}"\n        )\n    return [audio_by_id[audio_id] for audio_id in submission_ids]\n\n\ndef load_audio(audio_path):\n    audio, _ = librosa.load(\n        audio_path, sr=AUDIO_SAMPLE_RATE, mono=True, dtype=np.float32\n    )\n    if audio.size == 0 or not np.isfinite(audio).all():\n        raise ValueError(f"Invalid audio: {audio_path}")\n    return audio\n\n\ndef get_segment_starts(audio_length):\n    if audio_length <= SEGMENT_SAMPLES:\n        return [0]\n    last_start = audio_length - SEGMENT_SAMPLES\n    starts = list(range(0, last_start + 1, SEGMENT_SAMPLES))\n    if starts[-1] != last_start:\n        starts.append(last_start)\n    return starts\n\n\ndef extract_segment(audio, start):\n    if audio.size < SEGMENT_SAMPLES:\n        repeat_count = SEGMENT_SAMPLES // audio.size + 1\n        audio = np.tile(audio, repeat_count)\n        return audio[:SEGMENT_SAMPLES].astype(np.float32)\n    return audio[start : start + SEGMENT_SAMPLES].astype(np.float32, copy=False)\n\n\ndef prepare_panns_labels():\n    source = PANNS_DIR / "class_labels_indices.csv"\n    target = Path.home() / "panns_data" / "class_labels_indices.csv"\n    if not source.is_file():\n        raise FileNotFoundError(source)\n    target.parent.mkdir(parents=True, exist_ok=True)\n    shutil.copy2(source, target)\n\n\ndef load_panns_model(device):\n    prepare_panns_labels()\n    from panns_inference import AudioTagging, labels\n\n    model = AudioTagging(\n        checkpoint_path=str(PANNS_DIR / "Cnn14_mAP=0.431.pth"),\n        device=device.type,\n    )\n    label_groups = json.loads(\n        (PANNS_DIR / "component_labels.json").read_text(encoding="utf-8")\n    )\n    label_to_index = {label: index for index, label in enumerate(labels)}\n    voice_indices = [label_to_index[label] for label in label_groups["voice"]]\n    music_indices = [label_to_index[label] for label in label_groups["music"]]\n    return model, voice_indices, music_indices\n\n\ndef make_panns_segments(audio):\n    segments = []\n    for start in get_segment_starts(audio.size):\n        segment = extract_segment(audio, start)\n        segment = librosa.resample(\n            segment,\n            orig_sr=AUDIO_SAMPLE_RATE,\n            target_sr=PANNS_SAMPLE_RATE,\n            res_type="soxr_hq",\n        )\n        segments.append(segment.astype(np.float32))\n    return np.stack(segments)\n\n\ndef predict_presence(model, voice_indices, music_indices, audio):\n    predictions, _ = model.inference(make_panns_segments(audio))\n    voice_probability = float(predictions[:, voice_indices].max())\n    music_probability = float(predictions[:, music_indices].max())\n    return voice_probability, music_probability\n\n\ndef predict_presence_for_all_files(audio_files, device):\n    model, voice_indices, music_indices = load_panns_model(device)\n    scores = {}\n    for audio_path in tqdm(audio_files, desc="Presence", dynamic_ncols=True):\n        scores[audio_path.stem] = predict_presence(\n            model, voice_indices, music_indices, load_audio(audio_path)\n        )\n    del model\n    gc.collect()\n    if device.type == "cuda":\n        torch.cuda.empty_cache()\n    return scores\n\n\ndef load_htdemucs_model():\n    original_torch_load = torch.load\n\n    def load_trusted_checkpoint(*args, **kwargs):\n        kwargs.setdefault("weights_only", False)\n        return original_torch_load(*args, **kwargs)\n\n    torch.load = load_trusted_checkpoint\n    try:\n        model = get_model("htdemucs", repo=HTDEMUCS_DIR)\n    finally:\n        torch.load = original_torch_load\n    return model.cpu().eval()\n\n\ndef separate_voice_and_music(audio_path, model, device):\n    waveform = load_track(audio_path, model.audio_channels, model.samplerate).float()\n    mono_waveform = waveform.mean(0)\n    mean = mono_waveform.mean()\n    std = mono_waveform.std()\n    if float(std) < 1e-8:\n        length = round(waveform.shape[-1] * AUDIO_SAMPLE_RATE / model.samplerate)\n        silence = np.zeros(max(1, length), dtype=np.float32)\n        return silence, silence.copy()\n\n    normalized_waveform = (waveform - mean) / std\n    with torch.inference_mode():\n        sources = apply_model(\n            model,\n            normalized_waveform[None],\n            device=device,\n            shifts=0,\n            split=True,\n            overlap=0.25,\n            progress=False,\n        )[0]\n    sources = sources * std + mean\n\n    vocal_index = model.sources.index("vocals")\n    voice_audio = sources[vocal_index].mean(0, keepdim=True)\n    music_audio = torch.stack(\n        [sources[index] for index, name in enumerate(model.sources) if name != "vocals"]\n    ).sum(0).mean(0, keepdim=True)\n\n    voice_audio = torchaudio.functional.resample(\n        voice_audio, model.samplerate, AUDIO_SAMPLE_RATE\n    )[0]\n    music_audio = torchaudio.functional.resample(\n        music_audio, model.samplerate, AUDIO_SAMPLE_RATE\n    )[0]\n    return (\n        voice_audio.cpu().numpy().astype(np.float32),\n        music_audio.cpu().numpy().astype(np.float32),\n    )\n\n\nclass SSLBase(nn.Module):\n    def __init__(self):\n        super().__init__()\n        config = AutoConfig.from_pretrained(XLSR_CONFIG_DIR, local_files_only=True)\n        self.ssl = AutoModel.from_config(config)\n        self.hidden = self.ssl.config.hidden_size\n\n    def features(self, audio):\n        audio = (audio - audio.mean(1, keepdim=True)) / (\n            audio.std(1, keepdim=True) + 1e-5\n        )\n        return self.ssl(audio).last_hidden_state\n\n\nclass AttentivePool(nn.Module):\n    def __init__(self, dim):\n        super().__init__()\n        self.attention = nn.Sequential(\n            nn.Linear(dim, dim // 2), nn.Tanh(), nn.Linear(dim // 2, 1)\n        )\n\n    def forward(self, x):\n        weights = torch.softmax(self.attention(x), dim=1)\n        mean = (x * weights).sum(1)\n        std = (\n            (weights * (x - mean[:, None]).square())\n            .sum(1)\n            .clamp_min(1e-6)\n            .sqrt()\n        )\n        return torch.cat([mean, std], dim=-1)\n\n\nclass AttentionBlock(nn.Module):\n    def __init__(self, dim, dropout=0.1):\n        super().__init__()\n        self.norm1 = nn.LayerNorm(dim)\n        self.attention = nn.MultiheadAttention(\n            dim, 4, dropout=dropout, batch_first=True\n        )\n        self.norm2 = nn.LayerNorm(dim)\n        self.ff = nn.Sequential(\n            nn.Linear(dim, dim * 4),\n            nn.GELU(),\n            nn.Dropout(dropout),\n            nn.Linear(dim * 4, dim),\n        )\n\n    def forward(self, x):\n        z = self.norm1(x)\n        x = x + self.attention(z, z, z, need_weights=False)[0]\n        return x + self.ff(self.norm2(x))\n\n\nclass TCMBlock(nn.Module):\n    def __init__(self, dim, dropout=0.1):\n        super().__init__()\n        self.norm = nn.LayerNorm(dim)\n        self.temporal = nn.Conv1d(dim, dim, 7, padding=3, groups=dim)\n        self.gate = nn.Sequential(\n            nn.Linear(dim, dim // 8),\n            nn.SiLU(),\n            nn.Linear(dim // 8, dim),\n            nn.Sigmoid(),\n        )\n        self.attention = AttentionBlock(dim, dropout)\n\n    def forward(self, x):\n        z = self.norm(x)\n        gate = self.gate(z.mean(1)).unsqueeze(1)\n        x = x + self.temporal(z.transpose(1, 2)).transpose(1, 2) * gate\n        return self.attention(x)\n\n\nclass XLSRTCM(nn.Module):\n    def __init__(self, dropout=0.2, dim=256):\n        super().__init__()\n        self.ssl = SSLBase().ssl\n        self.hidden = self.ssl.config.hidden_size\n        self.projection = nn.Linear(self.hidden, dim)\n        self.tcm = nn.Sequential(TCMBlock(dim), TCMBlock(dim), TCMBlock(dim))\n        self.pool = AttentivePool(dim)\n        self.head = nn.Sequential(\n            nn.LayerNorm(dim * 2), nn.Dropout(dropout), nn.Linear(dim * 2, 2)\n        )\n\n    def features(self, audio):\n        audio = (audio - audio.mean(1, keepdim=True)) / (\n            audio.std(1, keepdim=True) + 1e-5\n        )\n        return self.ssl(audio).last_hidden_state\n\n    def forward(self, audio):\n        hidden = self.projection(self.features(audio))\n        return self.head(self.pool(self.tcm(hidden)))\n\n\ndef resolve_fake_label_index(checkpoint):\n    convention = checkpoint.get("label_convention", {"0": "fake", "1": "real"})\n    for raw_index, name in convention.items():\n        if str(name).strip().lower() == "fake":\n            return int(raw_index)\n    raise ValueError(f"No fake label in checkpoint label_convention: {convention}")\n\n\ndef load_xlsr_tcm_model(device):\n    checkpoint_path = XLSR_TCM_DIR / "best.pt"\n    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)\n    config = checkpoint.get("config", {})\n    if config.get("model") != "xlsr_tcm":\n        raise ValueError(\n            f"Expected an xlsr_tcm checkpoint, got config={config}"\n        )\n\n    model = XLSRTCM(dropout=float(config.get("dropout", 0.2)))\n    model.load_state_dict(checkpoint["model_state"], strict=True)\n    model.to(device).eval()\n    fake_label_index = resolve_fake_label_index(checkpoint)\n    eval_batch_size = max(1, int(config.get("eval_batch", 4)))\n    return model, fake_label_index, eval_batch_size\n\n\ndef calculate_rms(audio):\n    return float(np.sqrt(np.mean(np.square(audio, dtype=np.float64))))\n\n\ndef predict_fake(model, fake_label_index, audio, device, batch_size):\n    if calculate_rms(audio) < SILENCE_RMS:\n        return 0.0\n\n    segments = np.stack(\n        [extract_segment(audio, start) for start in get_segment_starts(audio.size)]\n    )\n    loader = DataLoader(\n        TensorDataset(torch.from_numpy(segments)),\n        batch_size=batch_size,\n        shuffle=False,\n        num_workers=0,\n        pin_memory=device.type == "cuda",\n    )\n    segment_scores = []\n    with torch.inference_mode():\n        for (segment_batch,) in loader:\n            segment_batch = segment_batch.to(device, non_blocking=True)\n            with torch.autocast(\n                device_type=device.type,\n                dtype=torch.float16,\n                enabled=device.type == "cuda",\n            ):\n                logits = model(segment_batch)\n            probabilities = torch.softmax(logits.float(), dim=-1)\n            segment_scores.extend(\n                probabilities[:, fake_label_index].cpu().numpy().tolist()\n            )\n    return float(max(segment_scores))\n\n\ndef combine_file_fake_score(voice_fake, music_fake, voice_present, music_present):\n    voice_risk = voice_present * voice_fake\n    music_risk = music_present * music_fake\n    return max(voice_risk, music_risk)\n\n\ndef predict_fake_scores_for_all_files(\n    audio_files, submission_rows, presence_scores, device\n):\n    model, fake_label_index, eval_batch_size = load_xlsr_tcm_model(device)\n    htdemucs_model = load_htdemucs_model()\n\n    for index, audio_path in enumerate(\n        tqdm(audio_files, desc="Components", dynamic_ncols=True)\n    ):\n        voice_audio, music_audio = separate_voice_and_music(\n            audio_path, htdemucs_model, device\n        )\n        voice_fake = predict_fake(\n            model, fake_label_index, voice_audio, device, eval_batch_size\n        )\n        music_fake = predict_fake(\n            model, fake_label_index, music_audio, device, eval_batch_size\n        )\n        voice_present, music_present = presence_scores[audio_path.stem]\n        file_fake = combine_file_fake_score(\n            voice_fake, music_fake, voice_present, music_present\n        )\n\n        row = submission_rows[index]\n        row["FILE_FAKE_PROB"] = round(float(np.clip(file_fake, 0.0, 1.0)), 10)\n        row["VOICE_FAKE_PROB"] = round(float(np.clip(voice_fake, 0.0, 1.0)), 10)\n        row["MUSIC_FAKE_PROB"] = round(float(np.clip(music_fake, 0.0, 1.0)), 10)\n        row["VOICE_PRESENT_PROB"] = round(\n            float(np.clip(voice_present, 0.0, 1.0)), 10\n        )\n        row["MUSIC_PRESENT_PROB"] = round(\n            float(np.clip(music_present, 0.0, 1.0)), 10\n        )\n\n    del model, htdemucs_model\n    gc.collect()\n    if device.type == "cuda":\n        torch.cuda.empty_cache()\n    return submission_rows\n\n\ndef save_submission(output_path, column_names, rows):\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n    with output_path.open("w", encoding="utf-8", newline="") as file:\n        writer = csv.DictWriter(file, fieldnames=column_names)\n        writer.writeheader()\n        writer.writerows(rows)\n\n\ndef main():\n    # DACON executes script.py without command-line arguments.\n    args = parse_arguments([])\n    device = select_device(args.device)\n    audio_files = find_audio_files(args.test_dir)\n    column_names, submission_rows = read_sample_submission(args.sample_submission)\n    audio_files = order_audio_files(audio_files, submission_rows)\n\n    presence_scores = predict_presence_for_all_files(audio_files, device)\n    submission_rows = predict_fake_scores_for_all_files(\n        audio_files, submission_rows, presence_scores, device\n    )\n    save_submission(args.output, column_names, submission_rows)\n\n    with args.output.open("r", encoding="utf-8", newline="") as file:\n        saved_rows = list(csv.DictReader(file))\n    if len(saved_rows) != len(submission_rows):\n        raise RuntimeError("submission row count validation failed")\n    print(f"Saved {len(saved_rows)} predictions to {args.output} on {device}")\n\n\nif __name__ == "__main__":\n    main()\n'

ast.parse(SCRIPT_SOURCE)
required_script_markers = [
    "class XLSRTCM",
    "def load_xlsr_tcm_model",
    "def separate_voice_and_music",
    'row["VOICE_FAKE_PROB"]',
    'row["MUSIC_FAKE_PROB"]',
    "softmax(logits.float(), dim=-1)",
]
missing_markers = [marker for marker in required_script_markers if marker not in SCRIPT_SOURCE]
if missing_markers:
    raise RuntimeError(f"script.py 필수 코드 누락: {missing_markers}")
if "df_arena" in SCRIPT_SOURCE.lower():
    raise RuntimeError("교체된 script.py에 DF-Arena 참조가 남아 있습니다.")
print("script.py AST and markers: OK")


In [ ]:
from transformers import AutoConfig

XLSR_MODEL_NAME = "facebook/wav2vec2-xls-r-300m"

stage_model = stage_root / "model"
stage_model.mkdir(parents=True, exist_ok=True)

shutil.copytree(panns_source, stage_model / "panns")
shutil.copytree(htdemucs_source, stage_model / "htdemucs")
(stage_model / "xlsr_tcm").mkdir(parents=True)
shutil.copy2(XLSR_TCM_CHECKPOINT, stage_model / "xlsr_tcm" / "best.pt")

# 같은 Colab 런타임의 Hugging Face cache를 우선 사용하고, 없으면 인터넷으로 config만 받습니다.
try:
    xlsr_config = AutoConfig.from_pretrained(
        XLSR_MODEL_NAME, local_files_only=True
    )
    print("XLS-R config: local cache")
except Exception:
    xlsr_config = AutoConfig.from_pretrained(XLSR_MODEL_NAME)
    print("XLS-R config: downloaded")
xlsr_config.save_pretrained(stage_model / "xlsr_config")

(stage_root / "script.py").write_text(SCRIPT_SOURCE, encoding="utf-8")

# 아래 패키지는 DACON 평가 서버에 지정 버전으로 기본 설치되어 있어 재설치하지 않습니다.
requirements_text = "\n".join([
    "# DACON evaluation-server preinstalled packages are used as-is.",
    "# torch==2.7.1+cu128",
    "# torchaudio==2.7.1+cu128",
    "# transformers==4.57.6",
    "# librosa==0.10.2.post1",
    "# soundfile==0.12.1",
    "# demucs==4.0.1",
    "# panns-inference==0.1.1",
    "# tqdm==4.66.4",
]) + "\n"
(stage_root / "requirements.txt").write_text(requirements_text, encoding="utf-8")

model_info = {
    "architecture": "PANNs Cnn14 + HTDemucs + user-trained XLS-R TCM",
    "dacon_baseline": "https://dacon.io/competitions/official/236749/codeshare/14153",
    "xlsr_backbone": "facebook/wav2vec2-xls-r-300m",
    "checkpoint_source": str(XLSR_TCM_CHECKPOINT),
    "checkpoint_epoch": checkpoint.get("epoch"),
    "label_convention": label_convention,
    "segment_samples": 64600,
    "sample_rate": 16000,
    "file_fake_fusion": "max(VP*VF, MP*MF)",
    "warning": "FoR-trained binary detector is also applied to accompaniment; it is not music-specialized.",
}
(stage_model / "model_info.json").write_text(
    json.dumps(model_info, ensure_ascii=False, indent=2), encoding="utf-8"
)

print("staging prepared:", stage_root)


## 5. `submit.zip` 생성 및 구조·용량 검증

최상위에는 오직 `model/`, `script.py`, `requirements.txt`만 존재합니다. 압축 파일 10GB 및 압축 해제 후 32GB 제한도 검사합니다.


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(
    OUTPUT_ZIP_PATH,
    "w",
    compression=zipfile.ZIP_STORED,
    allowZip64=True,
) as archive:
    for source_path in sorted(stage_root.rglob("*")):
        if source_path.is_file():
            archive.write(source_path, source_path.relative_to(stage_root).as_posix())

with zipfile.ZipFile(OUTPUT_ZIP_PATH) as archive:
    bad_members = archive.testzip()
    if bad_members is not None:
        raise RuntimeError(f"손상된 ZIP member: {bad_members}")
    infos = archive.infolist()
    names = [info.filename for info in infos]
    top_level = {name.split("/")[0] for name in names}
    if top_level != {"model", "script.py", "requirements.txt"}:
        raise RuntimeError(f"잘못된 최상위 구조: {sorted(top_level)}")

    required_members = {
        "script.py",
        "requirements.txt",
        "model/xlsr_tcm/best.pt",
        "model/xlsr_config/config.json",
        "model/panns/Cnn14_mAP=0.431.pth",
        "model/panns/class_labels_indices.csv",
        "model/panns/component_labels.json",
        "model/model_info.json",
    }
    missing_members = sorted(required_members - set(names))
    if missing_members:
        raise RuntimeError(f"ZIP 필수 파일 누락: {missing_members}")
    if any("df_arena" in name.lower() for name in names):
        raise RuntimeError("최종 ZIP에 DF-Arena 파일이 남아 있습니다.")

    uncompressed_bytes = sum(info.file_size for info in infos)

zip_bytes = OUTPUT_ZIP_PATH.stat().st_size
zip_gb = zip_bytes / 1024**3
uncompressed_gb = uncompressed_bytes / 1024**3
if zip_gb > 10:
    raise RuntimeError(f"ZIP 용량 제한 10GB 초과: {zip_gb:.2f}GB")
if uncompressed_gb > 32:
    raise RuntimeError(f"압축 해제 용량 제한 32GB 초과: {uncompressed_gb:.2f}GB")

print("saved:", OUTPUT_ZIP_PATH)
print(f"zip size: {zip_gb:.2f} GB")
print(f"uncompressed size: {uncompressed_gb:.2f} GB")
print("top-level:", sorted(top_level))
print("file count:", len(names))
for name in names:
    print(" -", name)


## 6. 제출 전 필수 확인

이 셀은 최종 `script.py`를 ZIP에서 다시 읽어 syntax와 오프라인 경로를 확인합니다. 실제 1,200개 평가 파일은 DACON 서버에만 있으므로 **리더보드 점수와 60분 내 완료 여부는 로컬 정적 검사로 보장할 수 없습니다.** 원본 베이스라인과 같은 PANNs·HTDemucs 흐름을 사용하면서 DF-Arena 1B보다 작은 XLS-R 300M으로 교체했지만, 제출 전 가능하면 DACON 제공 테스트 환경 또는 대표 길이 파일로 시간을 측정하세요.


In [ ]:
with zipfile.ZipFile(OUTPUT_ZIP_PATH) as archive:
    packaged_script = archive.read("script.py").decode("utf-8")

ast.parse(packaged_script)
checks = {
    "offline_xlsr_config": "local_files_only=True" in packaged_script,
    "dacon_test_path": 'Path("data") / "test"' in packaged_script,
    "dacon_output_path": 'Path("output") / "submission.csv"' in packaged_script,
    "five_outputs": all(column in packaged_script for column in [
        "FILE_FAKE_PROB", "VOICE_FAKE_PROB", "MUSIC_FAKE_PROB",
        "VOICE_PRESENT_PROB", "MUSIC_PRESENT_PROB",
    ]),
    "no_df_arena": "df_arena" not in packaged_script.lower(),
    "strict_checkpoint_load": "strict=True" in packaged_script,
    "fake_index_from_checkpoint": "resolve_fake_label_index" in packaged_script,
}
print(json.dumps(checks, indent=2))
if not all(checks.values()):
    raise RuntimeError(f"최종 제출 검사 실패: {checks}")

print("\n제출 파일 준비 완료:", OUTPUT_ZIP_PATH)


## 모델 출력 해석

- `VOICE_PRESENT_PROB`, `MUSIC_PRESENT_PROB`: 원본 베이스라인 PANNs가 원본 오디오에서 계산
- `VOICE_FAKE_PROB`: HTDemucs vocals를 XLS-R TCM에 넣은 `P(fake)`
- `MUSIC_FAKE_PROB`: HTDemucs accompaniment를 같은 XLS-R TCM에 넣은 `P(fake)`
- `FILE_FAKE_PROB`: `max(VOICE_PRESENT_PROB × VOICE_FAKE_PROB, MUSIC_PRESENT_PROB × MUSIC_FAKE_PROB)`

평가 서버의 FAKE 양성 클래스는 `1`이지만, 이는 제출 CSV의 의미입니다. 학습 checkpoint 내부 class index는 `0=fake`이므로 코드가 내부 index 0의 softmax를 꺼내 제출용 `*_FAKE_PROB`로 기록합니다.
